# Comparación, limpieza y conversión histórica

**Objetivo:** comparar los dos historiales sin mezclarlos y verificar la preparación para SQLite. Primero ejecutar `python -m scripts.eda_datasets` y `python -m scripts.preparar_datasets`; con caché, añadir `--offline`. Este notebook no modifica la base operativa.

In [1]:
from pathlib import Path
import sys,json
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'scripts/eda_datasets.py').exists())
sys.path.insert(0,str(ROOT))
quality=json.loads((ROOT/'reports/eda/quality_summary.json').read_text(encoding='utf-8'))
manifest=json.loads((ROOT/'reports/eda/preparacion_manifest.json').read_text(encoding='utf-8'))


## A. Comparación de estructura y cobertura

**Objetivo:** distinguir cantidad de transacciones de ejemplos semanales disponibles.

In [2]:
display(pd.DataFrame([{k:s[k] for k in ['file','rows','columns','date_min','date_max','calendar_bounded_weeks','invalid_dates','invalid_amounts','exact_duplicate_rows']} for s in quality['datasets']]))


,file,rows,columns,date_min,date_max,calendar_bounded_weeks,invalid_dates,invalid_amounts,exact_duplicate_rows
0,expense_data_1.csv,277,11,2021-11-21,2022-03-02,14,0,0,0
1,personal_transactions_dashboard_ready.xlsx,806,7,2018-01-01,2019-09-30,91,0,0,0


**Observado:** 806 frente a 277 movimientos y 91 frente a 14 semanas interiores.

**Interpretación:** el Excel tiene un historial más largo; ninguno certifica completitud o representatividad.

**Implicación:** mantener perfiles separados, incluso después de convertir monedas.

## B. Transformaciones y conservación de filas

**Objetivo:** verificar que la limpieza agrega metadatos y banderas sin borrar observaciones.

In [3]:
display(pd.DataFrame(manifest['datasets'])[['dataset_id','rows','conversion_status','review_rows','distinct_rates_used']])
print('Filas en staging:',manifest['sqlite_rows'])
print('Reglas:')
for rule in manifest['rules']:print('-',rule)


,dataset_id,rows,conversion_status,review_rows,distinct_rates_used
0,personal_finance,806,{'unidentified_currency': 806},806,0
1,my_expenses,277,{'converted': 277},78,86


Filas en staging: 1083
Reglas:
- Una fila de salida por fila original; sin borrados ni imputaciones.
- Fechas originales para FX. demo_date queda vacío en preparación.
- Moneda del Excel no verificada: COP nulo, no se asume USD.
- Tasa de la fecha o última anterior hasta siete días. Nunca tasa futura.
- Redondeo HALF_UP una vez a centavos COP mediante Decimal.
- Perfiles separados; los tipos ambiguos requieren revisión.
- Staging separado: no se carga en la base operativa.


**Observado:** 1.083 filas conservadas. Los campos dudosos quedan marcados y las cuentas tienen referencias seudonimizadas.

**Interpretación:** seudonimizar cuentas no anonimiza completamente descripciones e importes.

**Implicación:** los CSV preparados y la base staging permanecen en `data/processed/`, excluida de Git.

## C. Conversión histórica a COP

**Objetivo:** verificar fechas originales, precisión y procedencia de tasas. La fuente es una referencia blended de Frankfurter, no la tasa bancaria aplicada ni TRM certificada.

In [4]:
display(pd.DataFrame(manifest['fx_provenance']).T[['url','retrieved_at','response_sha256','source']])
for item in manifest['datasets']:
    prepared=pd.read_csv(ROOT/item['output_file'],dtype=str,keep_default_na=False)
    print(item['dataset_id'])
    display(prepared[['source_row','original_date','demo_date','original_amount_decimal','original_currency','amount_cop_minor','fx_rate_decimal','fx_reference_date','conversion_status']].head())
    print(prepared.conversion_status.value_counts())


,url,retrieved_at,response_sha256,source
INR,https://api.frankfurter.dev/v2/rates?from=2021...,2026-11-07T01:57:00.429131+00:00,e4cfe9e8494f7b5c7cfcd1946bcbbdeecbdfea83107096...,"Frankfurter v2: referencia blended, no tasa ba..."
USD,https://api.frankfurter.dev/v2/rates?from=2021...,2026-11-07T01:57:01.924168+00:00,568713489aa985f40dd3701aea51be5f9dbfa67dbc0d3a...,"Frankfurter v2: referencia blended, no tasa ba..."


personal_finance


,source_row,original_date,demo_date,original_amount_decimal,original_currency,amount_cop_minor,fx_rate_decimal,fx_reference_date,conversion_status
0,2,2018-01-01,,11.11,,,,,unidentified_currency
1,3,2018-01-02,,1247.44,,,,,unidentified_currency
2,4,2018-01-02,,24.22,,,,,unidentified_currency
3,5,2018-01-03,,2298.09,,,,,unidentified_currency
4,6,2018-01-04,,11.76,,,,,unidentified_currency


conversion_status
unidentified_currency    806
Name: count, dtype: int64
my_expenses


,source_row,original_date,demo_date,original_amount_decimal,original_currency,amount_cop_minor,fx_rate_decimal,fx_reference_date,conversion_status
0,2,2022-03-02,,50,INR,257185,51.437,2022-03-02,converted
1,3,2022-03-02,,300,INR,1543110,51.437,2022-03-02,converted
2,4,2022-03-01,,78,INR,405358,51.969,2022-03-01,converted
3,5,2022-03-01,,30,INR,155907,51.969,2022-03-01,converted
4,6,2022-03-01,,67,INR,348192,51.969,2022-03-01,converted


conversion_status
converted    277
Name: count, dtype: int64


**Observado:** los 277 movimientos del CSV se convierten con referencias históricas; los 806 del Excel quedan pendientes por moneda sin verificar.

**Interpretación:** COP nulo significa conversión no disponible, no importe cero. No se utilizan las fechas desplazadas de demostración.

**Implicación:** bloquear agregados COP incompletos; no rellenar con una tasa actual ni asumir USD.

## D. Conciliación matemática y almacenamiento

**Objetivo:** recomputar cada conversión usando Decimal y comprobar la conservación de filas.

In [5]:
from core.monedas import convertir
checked=0
for item in manifest['datasets']:
    prepared=pd.read_csv(ROOT/item['output_file'],dtype=str,keep_default_na=False)
    assert len(prepared)==item['rows']
    assert prepared.transaction_uid.is_unique
    for row in prepared.to_dict('records'):
        if row['conversion_status']=='converted':
            expected=convertir(int(row['original_amount_minor']),row['original_currency'],'COP',row['fx_rate_decimal'])
            assert expected==int(row['amount_cop_minor'])
            checked+=1
print('Conversiones verificadas:',checked)
print('La suma de todos los tipos de movimiento no equivale a gasto ni a ahorro.')


Conversiones verificadas: 277
La suma de todos los tipos de movimiento no equivale a gasto ni a ahorro.


**Observado:** se verifica la identidad de conversión fila a fila para las operaciones convertidas.

**Interpretación:** la comprobación aritmética no demuestra que la fuente externa represente la tasa bancaria efectiva.

**Implicación:** mantener fuente, fecha y respuesta guardadas; la integración en producción requiere resolver revisiones pendientes.

## E. Sesgos y viabilidad

**Objetivo:** delimitar qué conclusiones permite este material.

In [6]:
print('Sin entrenamiento de modelos ni resultados de precisión predictiva.')
print('Pendientes: moneda del Excel, conciliación de tarjeta, notas ambiguas, cobertura y representatividad.')


Sin entrenamiento de modelos ni resultados de precisión predictiva.


Pendientes: moneda del Excel, conciliación de tarjeta, notas ambiguas, cobertura y representatividad.


**Conclusión:** Excel para un experimento semanal exploratorio en su unidad de origen, condicionado a verificación semántica; CSV para pruebas y un análisis temporal corto. La conversión de moneda no adapta hábitos o precios históricos a Colombia actual.

No imputar ausencia de registros como gasto cero, no eliminar gastos extraordinarios por IQR y no seleccionar categorías con el periodo de prueba.

Fuentes verificadas: [documentación del dataset](https://www.kaggle.com/datasets/entrepreneurlife/personal-finance/data), metadatos guardados en `reports/eda/fuentes/`, [API histórica y política de tasas](https://frankfurter.dev/).